# The Same Model, Twice

**Topic:** State-space duality &nbsp;·&nbsp; **Difficulty:** Hard

A linear state-space model has two exactly equivalent forms: a recurrence that
costs `O(1)` per token and is perfect for generation, and a convolution with a
fixed kernel that computes the whole sequence in parallel and is perfect for
training. They are the same model, not an approximation of one another.

Implement both, check they agree to machine precision, and confirm the identity
breaks the moment you add a nonlinearity.

## The data (created in the next cell)

- Diagonal `A` with 16 complex-conjugate-safe real entries in `(0, 1)`.
- `B`, `C` real, sequence length 2048, input `x` `(2048,)`.
- Discretization is already applied (`A_bar`, `B_bar` are given directly).

## Your task

With **NumPy only**:

1. `recurrent(x)`: `h_t = A_bar * h_{t-1} + B_bar * x_t`, `y_t = C . h_t`.
2. `kernel`: `K_i = C . (A_bar^i * B_bar)` for `i = 0 .. T-1` (the sequence
   length, 2048).
3. `convolutional(x)`: apply `K` by FFT (zero-pad to `2T`, multiply, take the
   first `T` real entries).
4. `nonlinear_recurrent(x)`: the same recurrence with `h_t = tanh(...)`, which
   has **no** convolutional form.
5. Set `answers` to a dict with `"max_diff"` (recurrent vs convolutional),
   `"kernel_sum"` (the sum of the kernel entries, `K.sum()`),
   `"y_last"` (the final output of the linear recurrence),
   `"nonlinear_gap"`, the max difference between the nonlinear recurrence and
   the linear convolution, which should be large.

## Evaluation

Compared to the reference to a relative tolerance of 1e-8. The evaluation cell
requires the two linear forms to agree to `1e-9` and the nonlinear gap to exceed `1e-3`.

In [ ]:
import numpy as np

rng = np.random.default_rng(5005)
N, T = 16, 2048
A_bar = rng.uniform(0.80, 0.995, N)
B_bar = rng.normal(0, 1, N) / np.sqrt(N)
C = rng.normal(0, 1, N) / np.sqrt(N)
x = rng.normal(0, 1, T)
print(f"state size {N}, sequence {T}, decay range "
      f"{A_bar.min():.3f} to {A_bar.max():.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("max_diff", [], 4.787836793695988e-16, -8.016658879922729e-17, 1e-08),
    ("kernel_sum", [], 3.1763414217085506, -0.5318403020239436, 1e-08),
    ("y_last", [], 0.3957755693135879, -0.06626787563794537, 1e-08),
    ("nonlinear_gap", [], 0.8833907087488476, -0.14791318657848046, 1e-08),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "max_diff" in answers and "nonlinear_gap" in answers:
    _same = answers["max_diff"] < 1e-9
    _diff = answers["nonlinear_gap"] > 1e-3
    print(f"  {'two linear forms agree':<24} {'ok' if _same else 'wrong'}  "
          f"(max diff {answers['max_diff']:.2e})")
    print(f"  {'nonlinearity breaks it':<24} {'ok' if _diff else 'wrong'}  "
          f"(gap {answers['nonlinear_gap']:.4f})")
    _ok = _ok and _same and _diff

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Unrolling `h_t = A h_{t-1} + B x_t` from zero gives
`y_t = sum_i C A^i B x_{t-i}`, which is literally a convolution with kernel
`K_i = C A^i B`, with no approximation anywhere, which is why the two agree to
floating-point precision. The nonlinearity destroys it because `tanh` does not
distribute over the sum, so there is no fixed kernel to extract. That is exactly
why state-space models are *linear* recurrences with nonlinearities placed
between layers rather than inside the recurrence.

```python
import numpy as np

def recurrent(xs):
    h = np.zeros(N)
    out = np.empty(len(xs))
    for t, xt in enumerate(xs):
        h = A_bar * h + B_bar * xt
        out[t] = float(C @ h)
    return out

powers = A_bar[None, :] ** np.arange(T)[:, None]
kernel = (powers * B_bar[None, :]) @ C

def convolutional(xs):
    L = 2 * T
    Kf = np.fft.rfft(kernel, n=L)
    Xf = np.fft.rfft(xs, n=L)
    return np.fft.irfft(Kf * Xf, n=L)[:T]

def nonlinear_recurrent(xs):
    h = np.zeros(N)
    out = np.empty(len(xs))
    for t, xt in enumerate(xs):
        h = np.tanh(A_bar * h + B_bar * xt)
        out[t] = float(C @ h)
    return out

y_rec = recurrent(x)
y_conv = convolutional(x)
y_nl = nonlinear_recurrent(x)

answers = {
    "max_diff": float(np.abs(y_rec - y_conv).max()),
    "kernel_sum": float(kernel.sum()),
    "y_last": float(y_rec[-1]),
    "nonlinear_gap": float(np.abs(y_nl - y_conv).max()),
}
```